In [13]:
from pyspark import SparkContext

sc = SparkContext.getOrCreate()
sc.setLogLevel("ERROR")

In [14]:
def pagerank(file, beta=0.8, iterations=10):
    lines = sc.textFile(file)
    edges = lines.map(lambda x: tuple(map(int, x.split()))).distinct()
    links = edges.groupByKey().mapValues(list)
    n = links.count()
    ranks = links.map(lambda x: (x[0], 1.0 / n))
    for _ in range(iterations):
        contribs = links.join(ranks).flatMap(
            lambda x: [(dest, x[1][1] / len(x[1][0])) for dest in x[1][0]]
        )
        ranks = contribs.reduceByKey(lambda a, b: a + b) \
                        .mapValues(lambda r: (1 - beta)/n + beta * r)
    return ranks

In [15]:
def top_bottom(ranks):
    print('Top 5:\n', ranks.sortBy(lambda x: -x[1]).take(5))
    print("\n")
    print('Bottom 5:\n', ranks.sortBy(lambda x: x[1]).take(5))

In [16]:
print("-----------------------------------------------------------------------------------------------")
print('PAGERANK OUTPUT')
print("-----------------------------------------------------------------------------------------------")
print('\nSmall graph\n')
r_small = pagerank("small.txt")
top_bottom(r_small)
print("-----------------------------------------------------------------------------------------------")
print('\nFull graph\n')
r_whole = pagerank("whole.txt")
top_bottom(r_whole)
print("-----------------------------------------------------------------------------------------------")

-----------------------------------------------------------------------------------------------
PAGERANK OUTPUT
-----------------------------------------------------------------------------------------------

Small graph

Top 5:
 [(53, 0.03573119195317506), (14, 0.03417093570091833), (40, 0.03363005697429354), (1, 0.03000596562671799), (27, 0.02972015333896721)]


Bottom 5:
 [(85, 0.003409696601587542), (59, 0.0036698677992253945), (81, 0.0036953529509727067), (37, 0.003808201322261096), (89, 0.003922465435152918)]
-----------------------------------------------------------------------------------------------

Full graph

Top 5:
 [(263, 0.0020202932110331543), (537, 0.0019433405287774387), (965, 0.0019254505625726922), (243, 0.0018526346907954832), (285, 0.001827376100881599)]


Bottom 5:
 [(558, 0.0003286023969699676), (93, 0.0003513569318852105), (62, 0.00035314829788007246), (424, 0.00035481605111304054), (408, 0.0003877984430805082)]
------------------------------------------------